# ClinicalBridge — Data Exploration

This notebook explores the simulated dataset: 12 patient EHR records, RPM vital sign time series, and anamnesis self-reports that feed the multi-agent pipeline.

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

PATIENTS_DIR = Path("../data/patients")
RPM_DIR = Path("../data/rpm")
ANAMNESIS_DIR = Path("../data/anamnesis")
SCENARIOS_DIR = Path("../data/scenarios")

## 1. Patient Cohort Overview

In [2]:
patients = [json.loads(f.read_text()) for f in sorted(PATIENTS_DIR.glob("*.json"))]
print(f"Total patients: {len(patients)}")

rows = []
for p in patients:
    d = p["demographics"]
    rows.append({
        "patient_id": p["patient_id"],
        "name": d["name"],
        "age": d["age"],
        "sex": d["sex"],
        "conditions": len(p["problem_list"]),
        "medications": len(p["medications"]),
        "primary_dx": p["problem_list"][0]["label"] if p["problem_list"] else "N/A",
    })

df_cohort = pd.DataFrame(rows)
df_cohort

Total patients: 12


,patient_id,name,age,sex,conditions,medications,primary_dx
0,PT-001,Robert Harmon,64,M,3,3,Essential hypertension
1,PT-002,Margaret Liu,71,F,3,4,Essential hypertension
2,PT-003,David Okafor,58,M,3,2,Essential hypertension
3,PT-004,Susan Caldwell,69,F,2,2,Essential hypertension
4,PT-005,James Whitfield,55,M,3,4,Type 2 diabetes mellitus
5,PT-006,Patricia Endo,62,F,2,3,Type 2 diabetes mellitus
6,PT-007,Carlos Mendez,49,M,3,2,Type 2 diabetes mellitus
7,PT-008,Eleanor Bassett,74,F,4,5,"Heart failure, unspecified"
8,PT-009,Franklin Torres,68,M,4,5,"Heart failure, unspecified"
9,PT-010,Shirley Nakamura,77,F,3,4,"Heart failure, unspecified"


In [3]:
print("Age distribution:")
print(df_cohort["age"].describe().round(1))
print(f"\nSex: {df_cohort['sex'].value_counts().to_dict()}")
print(f"Avg conditions per patient: {df_cohort['conditions'].mean():.1f}")
print(f"Avg medications per patient: {df_cohort['medications'].mean():.1f}")

Age distribution:
count    12.0
mean     63.3
std       8.8
min      49.0
25%      57.2
50%      63.0
75%      69.5
max      77.0
Name: age, dtype: float64

Sex: {'M': 6, 'F': 6}
Avg conditions per patient: 2.9
Avg medications per patient: 3.3


## 2. Condition Distribution Across Cohort

In [4]:
from collections import Counter

all_conditions = []
for p in patients:
    for cond in p["problem_list"]:
        all_conditions.append(cond["label"])

condition_counts = Counter(all_conditions)
print("Top conditions in cohort:")
for cond, count in condition_counts.most_common(10):
    print(f"  {count:2d}x  {cond}")

Top conditions in cohort:
  10x  Essential hypertension
   6x  Type 2 diabetes mellitus
   3x  Heart failure, unspecified
   2x  Hyperlipidemia
   2x  Obesity
   2x  Chronic kidney disease, stage 3
   1x  Atherosclerotic heart disease
   1x  Osteoporosis
   1x  Nicotine dependence, cigarettes
   1x  Migraine


## 3. RPM Vital Sign Time Series

In [5]:
rpm_files = sorted(RPM_DIR.glob("*.csv"))
print(f"RPM files: {len(rpm_files)}")

df_pt001 = pd.read_csv(rpm_files[0], parse_dates=["timestamp"])
print(f"\nPT-001 RPM shape: {df_pt001.shape}")
print(f"Date range: {df_pt001['timestamp'].min().date()} → {df_pt001['timestamp'].max().date()}")
df_pt001.describe().round(1)

RPM files: 12

PT-001 RPM shape: (180, 7)
Date range: 2026-01-01 → 2026-01-31


,timestamp,systolic_bp,diastolic_bp,heart_rate,spo2,weight_kg,glucose_mgdl
count,180,180.0,180.0,180.0,180.0,180.0,180.0
mean,2026-01-16 04:00:00,143.1,90.0,73.9,97.5,88.5,117.1
min,2026-01-01 06:00:00,125.5,80.0,61.0,96.3,87.9,72.0
25%,2026-01-08 17:00:00,135.2,85.0,70.0,97.2,88.4,107.0
50%,2026-01-16 04:00:00,139.5,88.2,74.0,97.5,88.5,117.0
75%,2026-01-23 15:00:00,143.0,91.9,77.0,97.8,88.6,127.2
max,2026-01-31 02:00:00,199.5,118.9,86.0,98.6,89.1,173.0
std,NaN,14.5,7.8,4.9,0.5,0.2,15.8


In [6]:
# Blood pressure trend — PT-001 (missed_medication scenario)
# Baseline: systolic 110-145, but patient stopped Lisinopril → rising trend
print("PT-001 — Last 7 days systolic BP (4-hr intervals):")
last_7d = df_pt001.tail(42)  # 42 readings = 7 days × 6 per day
bp_stats = last_7d["systolic_bp"].agg(["mean", "min", "max"]).round(1)
print(bp_stats)
print(f"\nBaseline upper limit: 145.0")
print(f"Readings above baseline: {(last_7d['systolic_bp'] > 145).sum()} / {len(last_7d)}")

PT-001 — Last 7 days systolic BP (4-hr intervals):
mean    156.9
min     129.8
max     199.5
Name: systolic_bp, dtype: float64

Baseline upper limit: 145.0
Readings above baseline: 20 / 42


In [7]:
# Load all RPM files and compute per-patient summary stats
rpm_summaries = []
for f in rpm_files:
    pid = f.stem.split("_")[0]
    df = pd.read_csv(f, parse_dates=["timestamp"])
    rpm_summaries.append({
        "patient_id": pid,
        "readings": len(df),
        "days": (df["timestamp"].max() - df["timestamp"].min()).days + 1,
        "avg_systolic": df["systolic_bp"].mean().round(1),
        "max_systolic": df["systolic_bp"].max().round(1),
        "avg_hr": df["heart_rate"].mean().round(1),
        "avg_spo2": df["spo2"].mean().round(1),
    })

pd.DataFrame(rpm_summaries)

,patient_id,readings,days,avg_systolic,max_systolic,avg_hr,avg_spo2
0,PT-001,180,30,143.1,199.5,73.9,97.5
1,PT-002,180,30,128.4,140.0,68.1,98.1
2,PT-003,180,30,149.9,171.0,77.6,96.4
3,PT-004,180,30,125.9,134.7,62.4,98.4
4,PT-005,180,30,133.4,149.4,76.2,97.8
5,PT-006,180,30,147.8,169.7,80.3,97.0
6,PT-007,180,30,127.9,142.7,74.0,98.0
7,PT-008,180,30,118.3,135.2,76.2,97.2
8,PT-009,180,30,121.8,135.8,64.5,97.8
9,PT-010,180,30,110.3,121.7,71.6,97.0


## 4. Anamnesis Self-Reports

In [8]:
anamnesis_files = sorted(ANAMNESIS_DIR.glob("*.json"))
print(f"Anamnesis records: {len(anamnesis_files)}")

# Show PT-001 anamnesis (key scenario: patient stopped Lisinopril, reports cough)
pt001_anamnesis = json.loads(anamnesis_files[0].read_text())
print(f"\n--- PT-001 Anamnesis ---")
print(f"Chief complaint: {pt001_anamnesis['chief_complaint']}")
print(f"Medication adherence: {pt001_anamnesis['medication_adherence']}")
if pt001_anamnesis.get('symptom_diary'):
    print(f"Most recent diary entry: {pt001_anamnesis['symptom_diary'][-1]['entry']}")

Anamnesis records: 12

--- PT-001 Anamnesis ---
Chief complaint: I've been having bad headaches and my neck feels stiff. I feel flushed a lot.
Medication adherence: {'self_reported_compliance': '0%', 'stopped_date': 'approximately 2026-01-01', 'reason': 'Persistent dry cough - patient reports cough started within weeks of starting Lisinopril and has not resolved', 'notes': 'Patient has not informed doctor yet. Trying to manage without medication.'}
Most recent diary entry: Neck is stiff and I feel hot. My wife said I look red in the face. Still not taking that pill.


In [9]:
# Adherence overview across cohort
adherence_data = []
for f in anamnesis_files:
    a = json.loads(f.read_text())
    adh = a.get("medication_adherence", {})
    if isinstance(adh, dict):
        compliance = adh.get("self_reported_compliance", "N/A")
    else:
        compliance = str(adh)
    adherence_data.append({
        "patient_id": a["patient_id"],
        "chief_complaint": a.get("chief_complaint", "")[:60],
        "self_reported_compliance": compliance,
        "sensitive_flags": len(a.get("sensitive_flags", [])),
        "diary_entries": len(a.get("symptom_diary", [])),
    })

pd.DataFrame(adherence_data)

,patient_id,chief_complaint,self_reported_compliance,sensitive_flags,diary_entries
0,PT-001,I've been having bad headaches and my neck fee...,0%,0,3
1,PT-002,Routine check-in. Feeling well overall.,95%,0,1
2,PT-003,Blood pressure has been running high lately. S...,75%,0,2
3,PT-004,"Feeling great, just checking in as scheduled.",98%,0,1
4,PT-005,Blood sugar was high for a few days but seems ...,95%,0,4
5,PT-006,Feet feel tingly sometimes and I'm very thirst...,70%,0,2
6,PT-007,Weight has been creeping up and blood sugar fe...,90%,0,2
7,PT-008,Feeling a bit more short of breath than usual ...,100%,0,4
8,PT-009,Doing well. Walking daily as instructed. No ma...,95%,0,1
9,PT-010,Getting tired faster than before. A little mor...,95%,0,2


## 5. Test Scenario Inputs

In [10]:
scenarios = sorted([d.name for d in SCENARIOS_DIR.iterdir() if d.is_dir()])
print(f"Scenarios: {scenarios}\n")

for scenario in scenarios:
    alert_path = SCENARIOS_DIR / scenario / "input_alert.json"
    if alert_path.exists():
        alert = json.loads(alert_path.read_text())
        vals = alert["measured_values"]
        print(f"[{scenario}]")
        print(f"  Patient: {alert['patient_id']}  |  Category: {alert['alert_category']}")
        print(f"  Key values: { {k: v for k, v in vals.items()} }")
        print()

Scenarios: ['conflicting_data', 'false_alarm', 'incomplete_record', 'missed_medication', 'silent_deterioration']

[conflicting_data]
  Patient: PT-011  |  Category: URGENT
  Key values: {'systolic_bp': 164.8, 'diastolic_bp': 101.2, 'heart_rate': 80.0, 'spo2': 97.4, 'weight_kg': 91.2, 'glucose_mgdl': 162.0}

[false_alarm]
  Patient: PT-005  |  Category: URGENT
  Key values: {'systolic_bp': 136.2, 'diastolic_bp': 85.1, 'heart_rate': 74.0, 'spo2': 97.9, 'weight_kg': 98.2, 'glucose_mgdl': 214.0}

[incomplete_record]
  Patient: PT-012  |  Category: URGENT
  Key values: {'systolic_bp': 172.6, 'diastolic_bp': 106.3, 'heart_rate': 86.0, 'spo2': 97.1, 'weight_kg': 79.3, 'glucose_mgdl': 104.0}

[missed_medication]
  Patient: PT-001  |  Category: URGENT
  Key values: {'systolic_bp': 188.4, 'diastolic_bp': 112.7, 'heart_rate': 78.0, 'spo2': 97.2, 'weight_kg': 88.6, 'glucose_mgdl': 118.0}

[silent_deterioration]
  Patient: PT-008  |  Category: URGENT
  Key values: {'systolic_bp': 120.3, 'diastolic_

## 6. Data Quality Check

In [11]:
print("=== Data Completeness Check ===")
for p in patients:
    pid = p["patient_id"]
    has_rpm = (RPM_DIR / f"{pid}_vitals.csv").exists()
    has_anamnesis = (ANAMNESIS_DIR / f"{pid}_anamnesis.json").exists()
    labs = len(p.get("labs", []))
    notes = len(p.get("visit_notes", []))
    meds = len(p.get("medications", []))
    flag = " ← sparse (PT-012 transfer)" if pid == "PT-012" else ""
    print(f"{pid}: RPM={'Y' if has_rpm else 'N'} Anamnesis={'Y' if has_anamnesis else 'N'} labs={labs} notes={notes} meds={meds}{flag}")

=== Data Completeness Check ===
PT-001: RPM=Y Anamnesis=Y labs=3 notes=2 meds=3
PT-002: RPM=Y Anamnesis=Y labs=2 notes=1 meds=4
PT-003: RPM=Y Anamnesis=Y labs=1 notes=1 meds=2
PT-004: RPM=Y Anamnesis=Y labs=1 notes=1 meds=2
PT-005: RPM=Y Anamnesis=Y labs=3 notes=1 meds=4
PT-006: RPM=Y Anamnesis=Y labs=2 notes=1 meds=3
PT-007: RPM=Y Anamnesis=Y labs=2 notes=1 meds=2
PT-008: RPM=Y Anamnesis=Y labs=2 notes=2 meds=5
PT-009: RPM=Y Anamnesis=Y labs=2 notes=1 meds=5
PT-010: RPM=Y Anamnesis=Y labs=2 notes=1 meds=4
PT-011: RPM=Y Anamnesis=Y labs=3 notes=1 meds=5
PT-012: RPM=Y Anamnesis=Y labs=0 notes=1 meds=1 ← sparse (PT-012 transfer)
